# RAG vs Fine-Tuning Strategy Map

This notebook compares **RAG**, **fine-tuning-style behavior adaptation**, and **hybrid prompting** with real model calls when a provider key is configured. It does not run a real fine-tuning job; instead, it uses few-shot examples to approximate the kind of stable behavior that fine-tuning is often used to learn.

The original [RAG paper](https://arxiv.org/abs/2005.11401) frames retrieval as non-parametric memory. [LoRA](https://arxiv.org/abs/2106.09685) and [PEFT](https://huggingface.co/docs/peft/index) are common ways to adapt model behavior without retraining every parameter.


## Strategy Vocabulary

Think of the three strategies as answers to different change problems. **RAG** changes what the model can see during a request, so it is strongest when facts are fresh, citeable, or frequently updated. **Fine-tuning** changes model behavior, so it is strongest when the issue is tone, format, refusal style, or repeated task behavior. A **hybrid** system keeps retrieved evidence for factual grounding while using examples or adapted weights to make responses consistent.

A common mistake is to ask fine-tuning to memorize living policy. That creates maintenance risk: the model can sound confident while repeating stale information. Another common mistake is to ask RAG to fix style problems. Retrieval can provide facts, but it does not automatically make the answer concise, empathetic, or consistent.


## Definitions With A Concrete Support Example

**Parametric knowledge** is information stored inside model weights. It is useful for broad language patterns and common facts, but it is hard to audit and hard to update quickly. Example: a base model may know what an invoice is, but it should not be trusted to know the current refund policy for this course scenario.

**Non-parametric knowledge** is information supplied outside the model weights, usually through retrieval, tools, or prompt context. It is easier to update and cite. Example: the refund policy document can be edited today, embedded again, retrieved tomorrow, and cited in the answer.

**Behavior adaptation** means changing how the model responds rather than what facts it has access to. A model can learn to be concise, to refuse unsafe requests carefully, or to return JSON consistently. That is different from learning a specific policy. In the notebooks, few-shot examples stand in for this behavior adaptation so participants can compare the strategy before training anything.


### Exercise

A policy changes every week, but the response style is already acceptable. Which strategy should be tried first?


### Solution

Start with RAG. The main problem is freshness and citeable policy evidence, not learned response behavior.


## Visual Map: What Changes Across Strategies

Before writing prompts, separate the two places where an LLM system can get information: the model weights and the runtime context. RAG changes the runtime context; fine-tuning changes behavior stored in model parameters or adapter parameters; hybrid systems use both.

**Hugging Face Cookbook RAG diagram.** Source: [Hugging Face cookbook images](https://huggingface.co/datasets/huggingface/cookbook-images/blob/main/rag-diagram.png). This visual gives a compact end-to-end view of user question, retrieval, context assembly, and generation.

<img src="assets/hf-cookbook-rag-diagram.png" alt="Hugging Face RAG diagram" style="max-width: 760px; width: 100%; height: auto;">

---

The next visual zooms out from the runtime call and shows the broader workflow around ingestion and retrieval.

**Hugging Face Cookbook RAG workflow.** Source: [Hugging Face cookbook images](https://huggingface.co/datasets/huggingface/cookbook-images/blob/main/RAG_workflow.png). This workflow is helpful for strategy decisions because it shows that ingestion and query-time retrieval are operational stages that can be improved without retraining a model.

<img src="assets/hf-cookbook-rag-workflow.png" alt="Hugging Face RAG workflow" style="max-width: 760px; width: 100%; height: auto;">


### Exercise

Look at the two RAG visuals. Which part of the system changes at request time: model weights or retrieved context?


### Solution

The retrieved context changes at request time. The model weights stay the same in a RAG-only system.


## Load Dependencies And Shared Data

The setup is split into small cells so each responsibility is visible: imports first, then `.env` loading and fixture loading.


In [ ]:
import json
import os
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel

The notebooks share one small support-policy fixture. Keeping it shared makes the strategy comparisons fair: each notebook sees the same documents, requests, and style examples.


In [ ]:
# Load optional provider settings from .env without overwriting shell variables.
load_dotenv(dotenv_path=Path(".env"), override=False)

DATA_PATH = Path("data") / "strategy_cases.json"
with DATA_PATH.open(encoding="utf-8") as file:
    DATA = json.load(file)

# Keep each fixture role explicit so later cells do not mix evidence, examples, and evaluation.
DOCUMENTS = DATA["documents"]
REQUESTS = DATA["support_requests"]
STYLE_EXAMPLES = DATA["style_examples"]
SCENARIOS = DATA["strategy_scenarios"]

print(
    f"Loaded {len(DOCUMENTS)} documents, {len(REQUESTS)} requests, and {len(STYLE_EXAMPLES)} style examples."
)

## Configure The Provider

The notebooks use the OpenAI-compatible client style. Groq is tried first by default; Mistral is the fallback when the preferred provider has no key.


In [ ]:
class ProviderConfig(BaseModel):
    """Provider settings for one OpenAI-compatible chat endpoint."""

    provider: str
    model: str
    base_url: str
    api_key: str | None = None
    dry_run: bool = True

Provider selection is intentionally explicit. This makes fallback behavior easy to inspect before any model call happens.


In [ ]:
def provider_order(preferred_provider: str | None = None) -> list[str]:
    """Return the preferred provider followed by the other supported provider."""
    requested = (preferred_provider or os.getenv("LLM_PROVIDER") or "groq").lower()

    if requested == "groq":
        return ["groq", "mistral"]
    if requested == "mistral":
        return ["mistral", "groq"]

    # Unknown values keep dry mode readable instead of failing during setup.
    return [requested]

The loader returns dry mode when no key is available. Dry mode is part of the teaching design: notebooks stay runnable during review and in classrooms without credentials.


Each provider has a tiny environment-to-config adapter. Keeping this separate makes the fallback loader shorter and easier to read.


In [ ]:
def provider_config_from_env(provider_name: str) -> ProviderConfig | None:
    """Return live provider settings when that provider has an API key."""
    if provider_name == "groq" and os.getenv("GROQ_API_KEY"):
        return ProviderConfig(
            provider="groq",
            model=os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile"),
            base_url="https://api.groq.com/openai/v1",
            api_key=os.getenv("GROQ_API_KEY"),
            dry_run=False,
        )

    if provider_name == "mistral" and os.getenv("MISTRAL_API_KEY"):
        return ProviderConfig(
            provider="mistral",
            model=os.getenv("MISTRAL_MODEL", "mistral-small-latest"),
            base_url="https://api.mistral.ai/v1",
            api_key=os.getenv("MISTRAL_API_KEY"),
            dry_run=False,
        )

    return None

The loader checks providers in order and falls back to dry mode when no key is available.


In [ ]:
def load_provider_config(preferred_provider: str | None = None) -> ProviderConfig:
    """Load live provider settings, or return a deterministic dry-run config."""
    # Check providers in priority order so fallback behavior is deterministic.
    for provider_name in provider_order(preferred_provider):
        config = provider_config_from_env(provider_name)
        if config is not None:
            return config

    # No key was found, so keep the notebook runnable without network calls.
    fallback_model = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")
    return ProviderConfig(
        provider=provider_order(preferred_provider)[0],
        model=fallback_model,
        base_url="dry-run",
    )

In [ ]:
CONFIG = load_provider_config()
print(
    f"Provider: {CONFIG.provider} | model: {CONFIG.model} | dry run: {CONFIG.dry_run}"
)

if CONFIG.dry_run:
    print("Set GROQ_API_KEY or MISTRAL_API_KEY in .env to run live model calls.")

## Call The Model Or Use Dry Mode

The next cells isolate client creation, request assembly, and fallback behavior. That keeps provider plumbing separate from the lesson logic.


In [ ]:
def get_client(config: ProviderConfig) -> OpenAI | None:
    """Create a client only when a live API key is configured."""
    if config.dry_run or config.api_key is None:
        return None

    return OpenAI(api_key=config.api_key, base_url=config.base_url)

Request parameters are built in one small helper so response formatting can be added only where a notebook needs structured JSON.


In [ ]:
def completion_kwargs(
    messages: list[dict[str, str]],
    *,
    temperature: float,
    response_format: dict[str, str] | None,
) -> dict[str, Any]:
    """Build provider request arguments for chat completions."""
    kwargs: dict[str, Any] = {
        "model": CONFIG.model,
        "messages": messages,
        "temperature": temperature,
    }

    # JSON mode is optional because not every provider/model combination needs it.
    if response_format is not None:
        kwargs["response_format"] = response_format

    return kwargs

`call_model` is the single gateway used by lesson cells. If live credentials are missing or a live call fails, the notebook returns a deterministic teaching response.


Live completion execution is separated from dry-mode routing so the failure handling is easy to follow.


In [ ]:
def live_completion_content(
    messages: list[dict[str, str]],
    *,
    temperature: float,
    response_format: dict[str, str] | None,
) -> str:
    """Run the live chat completion and return message content."""
    client = get_client(CONFIG)
    assert client is not None

    completion = client.chat.completions.create(
        **completion_kwargs(
            messages, temperature=temperature, response_format=response_format
        )
    )
    return completion.choices[0].message.content or ""

`call_model` now only decides between dry output, live output, and live-call fallback.


In [ ]:
def call_model(
    messages: list[dict[str, str]],
    *,
    dry_response: str,
    temperature: float = 0.2,
    response_format: dict[str, str] | None = None,
) -> str:
    """Call the configured provider, or return deterministic dry output."""
    # Dry mode is the expected path for participants without provider credentials.
    if CONFIG.dry_run:
        return dry_response

    try:
        return live_completion_content(
            messages, temperature=temperature, response_format=response_format
        )
    except Exception as error:
        print(f"Live provider call failed: {error}")
        print("Using deterministic dry response so the notebook remains runnable.")
        return dry_response

## Three Prompting Conditions

We will ask the same user question three ways:

- baseline: no retrieved evidence and no style examples;
- retrieval-aware: retrieved policy context is supplied;
- style-aware: response examples are supplied, but no fresh policy context is supplied.

The contrast shows why RAG and fine-tuning solve different problems.


## How To Read The Comparison

The three prompt variants are not meant to compete as polished production prompts. They are controlled experiments. The baseline tells us what the model does with only a general instruction. The retrieval-aware version tells us how much better the answer becomes when policy evidence is added. The style-aware version tells us whether examples can steer tone and structure without adding fresh facts.

A practical rule of thumb: if the model knows the tone but misses the policy, add retrieval. If it cites correctly but sounds inconsistent, improve examples or prompts. If both evidence and style matter, test a hybrid path before planning a real fine-tuning project.


The next comparison intentionally changes one ingredient at a time. The baseline has only the user question, the retrieval-aware prompt adds policy evidence, and the style-aware prompt adds examples without policy context. This makes the failure modes easier to name: missing facts, missing citations, or inconsistent behavior.


In [ ]:
# Exercise: name the variant that mainly tests behavior adaptation.
# behavior_variant = ...


In [ ]:
# Solution
behavior_variant = "style_aware"
print(behavior_variant)

Few-shot prompting turns approved examples into alternating user and assistant messages. The examples are not new model weights; they are context supplied at request time.


## Build Few-Shot Style Messages

Few-shot messages make behavior examples visible in the prompt. They do not update model weights.


In [ ]:
def style_messages(examples: list[dict[str, str]]) -> list[dict[str, str]]:
    """Convert approved examples into alternating user and assistant messages."""
    messages: list[dict[str, str]] = []

    for example in examples:
        # The user turn shows the incoming request pattern.
        messages.append({"role": "user", "content": example["user"]})
        # The assistant turn shows the response style we want to imitate.
        messages.append({"role": "assistant", "content": example["assistant"]})

    return messages


def compact_style_examples(examples: list[dict[str, str]]) -> str:
    """Create a readable style summary for prompts and reports."""
    lines = []
    for example in examples:
        lines.append(
            f"- intent={example['intent']} tone={example['tone']}: {example['assistant']}"
        )
    return "\n".join(lines)

Start with one request so the prompt differences stay easy to compare.


In [ ]:
question = REQUESTS[0]["question"]
retrieved = [DOCUMENTS[0]]
style_seed = STYLE_EXAMPLES[:2]

The baseline has no domain evidence and no style examples.


In [ ]:
baseline_messages = [
    {"role": "system", "content": "You are a concise support assistant."},
    {"role": "user", "content": question},
]

The retrieval-aware version adds one citeable policy document to the user message.


In [ ]:
retrieval_messages = [
    {
        "role": "system",
        "content": "Answer only from the provided policy context. Cite source IDs in square brackets.",
    },
    {
        "role": "user",
        "content": f"Policy context:\n[refund-policy] {retrieved[0]['text']}\n\nQuestion: {question}",
    },
]

The style-aware version adds examples but no fresh policy context, which is why it can imitate tone but may miss facts.


In [ ]:
style_messages_only = [
    {
        "role": "system",
        "content": "Match the style of the examples. Do not invent policy details.",
    },
    *style_messages(style_seed),
    {"role": "user", "content": question},
]

Collect the variants in one dictionary so the same model-call loop can compare them.


In [ ]:
prompt_variants = {
    "baseline": baseline_messages,
    "retrieval_aware": retrieval_messages,
    "style_aware": style_messages_only,
}

for name, messages in prompt_variants.items():
    print(name, "messages=", len(messages))

In [ ]:
# Exercise: complete the list with the prompt variant that contains retrieved policy text.
# evidence_variants = [...]


In [ ]:
# Solution
evidence_variants = ["retrieval_aware"]
print(evidence_variants)

## Call The Model

In dry mode, the function returns deterministic teaching outputs. With `GROQ_API_KEY` or `MISTRAL_API_KEY`, the same cell calls the live model.


In [ ]:
dry_outputs = {
    "baseline": "A refund may be possible, but I need the invoice and payment details.",
    "retrieval_aware": "Duplicate payments can be refunded when both payments belong to the same invoice. Provide the invoice number and both payment references within 30 days. [refund-policy]",
    "style_aware": "Ask for the invoice number and both payment references, then explain the next step in a concise support tone.",
}

responses = {}
for name, messages in prompt_variants.items():
    responses[name] = call_model(messages, dry_response=dry_outputs[name])
    print()
    print(name.upper())
    print(responses[name])

## Interpret The Difference

The retrieval-aware answer can cite a specific source because the prompt supplied source text. The style-aware answer can mimic response behavior, but it still cannot prove the current refund rule unless policy evidence is present.


### Exercise

Which response should be easiest to audit: baseline, retrieval-aware, or style-aware?


### Solution

The retrieval-aware response is easiest to audit because it asks the model to cite source IDs from the provided policy context.


In [ ]:
mfa_question = REQUESTS[2]["question"]
mfa_context = next(
    document for document in DOCUMENTS if document["doc_id"] == "mfa-reset"
)
messages = [
    {
        "role": "system",
        "content": "Answer only from policy context and cite source IDs.",
    },
    {
        "role": "user",
        "content": f"Policy context:\n[mfa-reset] {mfa_context['text']}\n\nQuestion: {mfa_question}",
    },
]
print(
    call_model(
        messages,
        dry_response="MFA can be reset only after identity verification by support. Do not send identity documents or backup codes through chat. [mfa-reset]",
    )
)